<a href="https://colab.research.google.com/github/danila-kopitayko/DataAnalysis/blob/main/%D0%9E%D1%82%D1%82%D0%BE%D0%BA%20%D0%BA%D0%BB%D0%B8%D0%B5%D0%BD%D1%82%D0%BE%D0%B2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Предсказание оттока клиентов**

Есть данные о клиентах телеком компании как пол, возраст, длительность сотрудничества, потраченные деньги на подписку и т.д. Необходимо определить, уйдет ли клиент или нет. Для этого используются логистическая регрессия, которая обучается на наборе данных для обучения train, и предсказывает, уйдет ли клиент на датасете test.

## Загрузка данных. Исследование.

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

In [ ]:
!gdown 1ERwQ5odiK1Zvi1LtjpkzCMUswYsAX8_K  # train.csv
!gdown 1fGw_-RFwvn_LEdt91Jq-7A-wzG6mmH8r  # test.csv
!gdown 199Mt4OYZNaelT83U-HGDsEYs2YcUGQ6y  # submission.csv

In [ ]:
data = pd.read_csv('./train.csv')

test = pd.read_csv('test.csv')

In [ ]:
data.info()

In [ ]:
# Числовые признаки
num_cols = [
    'ClientPeriod',
    'MonthlySpending',
    'TotalSpent'
]

# Категориальные признаки
cat_cols = [
    'Sex',
    'IsSeniorCitizen',
    'HasPartner',
    'HasChild',
    'HasPhoneService',
    'HasMultiplePhoneNumbers',
    'HasInternetService',
    'HasOnlineSecurityService',
    'HasOnlineBackup',
    'HasDeviceProtection',
    'HasTechSupportAccess',
    'HasOnlineTV',
    'HasMovieSubscription',
    'HasContractPhone',
    'IsBillingPaperless',
    'PaymentMethod'
]

feature_cols = num_cols + cat_cols
target_col = 'Churn'

In [ ]:
test.info()

In [ ]:
data.sample(5)

В нашем распоряжении таблица с данными о клиентах телеком компании. У нас есть 20 признаков для каждого клиента, всего клиентов 5282.

Признак TotalSpent переведем в тип float.

In [ ]:
data[data['TotalSpent'] == ' ']['TotalSpent'].count()

In [ ]:
test[test['TotalSpent'] == ' ']['TotalSpent'].count()

Всего 9 строчек с незаполненными данными в признаке TotalSpent, поэтому удалим их.

В случае с тестовой выборкой заполним медианным значением, чтобы оставить нужный размер таблицы.

In [ ]:
data.loc[data['TotalSpent'] == ' ', 'TotalSpent'] = pd.NA

data.dropna(inplace=True)

test.loc[test['TotalSpent'] == ' ', 'TotalSpent'] = 0

In [ ]:
data['TotalSpent'] = data['TotalSpent'].astype('float')
test['TotalSpent'] = test['TotalSpent'].astype('float')

In [ ]:
test.loc[test['TotalSpent']==0,'TotalSpent'] = test['TotalSpent'].median()

In [ ]:
data['HasPartner'].unique()

In [ ]:
data['HasChild'].unique()

In [ ]:
data['HasPhoneService'].unique()

In [ ]:
data['HasMultiplePhoneNumbers'].unique()

In [ ]:
data['HasInternetService'].unique()

In [ ]:
data['HasOnlineSecurityService'].unique()

In [ ]:
data['HasOnlineBackup'].unique()

In [ ]:
data['HasDeviceProtection'].unique()

In [ ]:
data['HasTechSupportAccess'].unique()

In [ ]:
data['HasOnlineTV'].unique()

In [ ]:
data['HasMovieSubscription'].unique()

In [ ]:
data['HasContractPhone'].unique()

In [ ]:
data['IsBillingPaperless'].unique()

In [ ]:
data['PaymentMethod'].unique()

In [ ]:
data['Churn'].unique()

Явных пропусков нет, неявные были в столбце TotalSpent, мы их уже удалили.

## Анализ данных

In [ ]:
data[num_cols].hist()
plt.show()

In [ ]:
data['ClientPeriod'].plot.box()
plt.show()

In [ ]:
data['TotalSpent'].plot.box()
plt.show()

In [ ]:
data['MonthlySpending'].plot.box()
plt.show()

In [ ]:
data[num_cols].describe()

Выбросов ни в одной из численных перменных нет.

In [ ]:
for col in cat_cols:
  display(data[col].value_counts())

In [ ]:
fig = plt.figure(0,figsize=(7.5,7.5))
fig.set_figheight(25)
fig.set_figwidth(25)
plots = []
for i in range(4):
    for j in range(4):
        ax = plt.subplot2grid((4,4), (i,j))
        ax.pie(data[cat_cols[i*4+j]].value_counts().reset_index().sort_values(by=['count'])['count'], autopct='%1.1f%%')
        ax.set_title(cat_cols[i*4+j])
        ax.legend(data[cat_cols[i*4+j]].value_counts().reset_index().sort_values(by=['count'])[cat_cols[i*4+j]].to_list())

plt.subplots_adjust(left=0.1, right=0.9,
                    top=0.9, bottom=0.1,
                    wspace=0.7, hspace=0.7)

plt.show()

In [ ]:
data['Churn'].hist()
plt.title('Churn')
plt.show()

In [ ]:
data[data['Churn'] == 0]['Churn'].count() / data[data['Churn'] == 1]['Churn'].count()

Есть небольшой дисбаланс, но классы различаются в почти три раза по количеству, поэтому оставим, как есть.

## Применение моделей

In [ ]:
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import make_pipeline

In [ ]:
data_new = data.copy()
data_new.head()

In [ ]:
lenc = LabelEncoder()

for col in cat_cols:
  test[col] = lenc.fit_transform(test[col])
  data_new[col] = lenc.fit_transform(data_new[col])

In [ ]:
data.head()

In [ ]:
enc = OneHotEncoder(sparse_output=False)

encoded = enc.fit_transform(data_new[cat_cols])
test_encoded = enc.fit_transform(test[cat_cols])

In [ ]:
encoded.shape

In [ ]:
one_hot_df = pd.DataFrame(encoded, columns=enc.get_feature_names_out(cat_cols))

one_hot_test = pd.DataFrame(test_encoded, columns=enc.get_feature_names_out(cat_cols))

In [ ]:
data_encoded = pd.concat([data_new[num_cols].reset_index(drop=True), one_hot_df.reset_index(drop=True)], axis=1)

test_ready = pd.concat([test[num_cols].reset_index(drop=True), one_hot_test.reset_index(drop=True)], axis=1)

In [ ]:
X=data_encoded.values
y=data_new['Churn'].values

In [ ]:
scaler = StandardScaler()
clf = LogisticRegression()

parameters={'logisticregression__C':[100,10,1,0.1,0.01,0.001]}
pipeline =  make_pipeline(scaler,clf)
grid_search = GridSearchCV(pipeline,param_grid = parameters, scoring = 'roc_auc',cv=5, refit=True)

grid_search.fit(X,y)

In [ ]:
scores_df = pd.DataFrame(grid_search.cv_results_).sort_values(by='rank_test_score')
scores_df

In [ ]:
print("Best cross-validation score: {:.2f}".format(grid_search.best_score_))

In [ ]:
best_row = scores_df.iloc[0,:]

In [ ]:
best_param = best_row['param_logisticregression__C']

In [ ]:
best_param

In [ ]:
!pip install catboost

In [ ]:
import catboost

In [ ]:
categories = np.array([3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(data.drop(columns={'Churn'}).values,data['Churn'].values,train_size=0.8,random_state=42)

In [ ]:
boost = catboost.CatBoostClassifier(n_estimators = 200, logging_level='Silent', cat_features = categories, eval_metric = 'AUC')

boost.grid_search({'l2_leaf_reg': np.linspace(0, 1, 20)}, X=X_train, y=y_train,plot=True, refit=True, verbose=False)

In [ ]:
y_train_predicted = boost.predict_proba(X_train)[:, 1]
y_test_predicted = boost.predict_proba(X_test)[:, 1]

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

In [ ]:
train_auc = roc_auc_score(y_train, y_train_predicted)
test_auc = roc_auc_score(y_test, y_test_predicted)

plt.figure(figsize=(10,7))
plt.plot(*roc_curve(y_train, y_train_predicted)[:2], label='train AUC={:.4f}'.format(train_auc))
plt.plot(*roc_curve(y_test, y_test_predicted)[:2], label='test AUC={:.4f}'.format(test_auc))
legend_box = plt.legend(fontsize='large', framealpha=1).get_frame()
legend_box.set_facecolor("white")
legend_box.set_edgecolor("black")
plt.plot(np.linspace(0,1,100), np.linspace(0,1,100))
plt.show()

CatBoost получается сравним с линейной регрессией по качеству предсказаний: у обеих моделей качество примерно 0.84.

In [ ]:
X_test = pd.read_csv('./test.csv')
submission = pd.read_csv('./submission.csv',index_col='Id')

submission['Churn'] = grid_search.predict(test_ready.values)
submission.to_csv('./my_submission.csv')